# High-Throughput Serving Architectures

**Duration:** 5 min

This module delves into the intricacies of high-throughput serving architectures, focusing on techniques and tools that enable efficient and scalable deployment of machine learning models. Understanding these concepts is crucial for optimizing performance, reducing latency, and managing costs in production environments.

## vLLM and TensorRT for Model Serving

vLLM and TensorRT are powerful tools for optimizing the inference of machine learning models. vLLM (Very Large Language Model) is designed to handle extremely large models efficiently, while TensorRT is a deep learning inference optimizer and runtime that maximizes performance on NVIDIA GPUs. Together, they provide a robust framework for high-throughput serving by leveraging hardware acceleration and efficient model execution.

In [ ]:
import torch
import tensorrt as trt

# Load a pre-trained PyTorch model
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
model.eval()

# Convert the model to TensorRT
builder = trt.Builder(trt.Logger(trt.Logger.WARNING))
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
profile = builder.create_optimization_profile()
config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)  # 4GB

# Add the input and output tensors to the network
input_tensor = network.add_input('input', trt.float32, (1, 3, 224, 224))
profile.set_shape(input_tensor.name, (1, 3, 224, 224), (4, 3, 224, 224), (8, 3, 224, 224))
network.mark_input(input_tensor)
network.mark_output(input_tensor)

# Build the engine
engine = builder.build_engine(network, config)

# Save the engine to a file
with open('resnet50.engine', 'wb') as f:
    f.write(engine.serialize())

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/production-inference/mod-7.ipynb)

```
Engine saved to 'resnet50.engine'
```

## Batching and Load Balancing

Batching and load balancing are essential techniques for achieving high-throughput in model serving. Batching involves processing multiple inference requests together, which can significantly reduce the overhead per request. Load balancing ensures that the incoming requests are distributed evenly across multiple servers or GPUs, preventing any single resource from becoming a bottleneck and ensuring consistent performance.

In [ ]:
from torch.utils.data import DataLoader
import torch

# Define a simple dataset
class RandomDataset(torch.utils.data.Dataset):
    def __init__(self, size, num_features):
        self.len = size
        self.data = torch.randn(size, num_features)
    def __getitem__(self, index):
        return self.data[index]
    def __len__(self):
        return self.len

# Create a dataset and dataloader with batching
dataset = RandomDataset(1000, 10)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Simulate model inference
model = torch.nn.Linear(10, 1)
model.eval()

for batch in dataloader:
    outputs = model(batch)
    print(outputs)

> **💡 Tip:** When implementing batching, ensure that the batch size is optimized for your specific hardware and model to avoid underutilization or overloading.

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is the primary benefit of using vLLM and TensorRT for model serving?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386962048" value="0">
      <span>Reduced model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386962048" value="1">
      <span>Increased model accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386962048" value="2">
      <span>Efficient hardware utilization</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386962048" value="3">
      <span>Simplified model training</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is the main purpose of batching in high-throughput serving?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386950592" value="0">
      <span>To increase model complexity</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386950592" value="1">
      <span>To reduce the number of model parameters</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386950592" value="2">
      <span>To process multiple inference requests together</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4386950592" value="3">
      <span>To simplify data preprocessing</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>